# PHẦN 3: XỬ LÝ ĐẶC TRƯNG (FEATURE ENGINEERING) 
## Định nghĩa vấn đề

### Mô tả
- phát hiện và phân loại các cuộc tấn công mạng (intrusion detection) dựa trên tập dữ liệu lưu lượng mạng và hành vi người dùng.  
- dữ liệu ban đầu bao gồm 1 file: cybersecurity_intrusion_data.csv.  
- link tập dữ liệu: https://www.kaggle.com/datasets/dnkumars/cybersecurity-intrusion-detection-dataset  

### Mục tiêu
- Kiểm tra và xử lý triệt để các giá trị thiếu (missing values) trong tập dữ liệu.  
- Phát hiện outlier, xử lý phân phối lệch (skewness) và chuẩn hóa (scaling) các đặc trưng số.  
- Chuyển đổi các biến phân loại thành dạng số phù hợp cho mô hình.
- Xây dựng các đặc trưng tương tác có giá trị như tỷ lệ đăng nhập thất bại, điểm rủi ro IP, phân nhóm thời lượng phiên.
- Tạo tập dữ liệu cuối cùng không còn giá trị thiếu, tất cả các cột đều ở dạng số, sẵn sàng cho huấn luyện mô hình.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)


### 0. ĐỌC DỮ LIỆU ĐÃ QUA EDA

In [29]:
# Đọc dữ liệu đã được làm sạch từ Phần 1 (chưa qua xử lý đặc trưng)
df = pd.read_parquet("../data_processed/cleaned_data.parquet")
print(f"\n Dữ liệu đầu vào: {df.shape[0]} hàng, {df.shape[1]} cột")
print(f"Các cột hiện có: {list(df.columns)}")

# Tạo bản sao để xử lý (giữ nguyên dữ liệu gốc)
df_original = df.copy()
df_processed = df.copy()



 Dữ liệu đầu vào: 7571 hàng, 10 cột
Các cột hiện có: ['network_packet_size', 'protocol_type', 'login_attempts', 'session_duration', 'encryption_used', 'ip_reputation_score', 'failed_logins', 'browser_type', 'unusual_time_access', 'attack_detected']


### 1. PHÂN LOẠI CÁC ĐẶC TRƯNG

In [30]:
# 1.1. Xác định loại dữ liệu của từng cột
feature_types = {
    'ID features': [],           # Cột định danh (sẽ bỏ)
    'Numerical features': [],    # Biến số liên tục
    'Categorical features': [],  # Biến phân loại (danh mục)
    'Binary features': [],       # Biến nhị phân (0/1)
    'Target feature': []         # Biến mục tiêu
}

for col in df_processed.columns:
    if col == 'attack_detected':
        feature_types['Target feature'].append(col)
    elif col == 'session_id':
        feature_types['ID features'].append(col)
    elif df_processed[col].dtype in ['int64', 'float64']:
        unique_vals = df_processed[col].nunique()
        if unique_vals == 2 and set(df_processed[col].unique()).issubset({0, 1}):
            feature_types['Binary features'].append(col)
        else:
            feature_types['Numerical features'].append(col)
    else:
        feature_types['Categorical features'].append(col)

# In kết quả phân loại
print("\n PHÂN LOẠI ĐẶC TRƯNG:")
for ftype, cols in feature_types.items():
    if cols:
        print(f"\n   {ftype}: {len(cols)} cột")
        print(f"     {cols}")

# 1.2. Xóa cột ID (không có giá trị cho mô hình)
if feature_types['ID features']:
    print(f"\n Đã xóa cột ID: {feature_types['ID features']}")
    df_processed = df_processed.drop(columns=feature_types['ID features'])




 PHÂN LOẠI ĐẶC TRƯNG:

   Numerical features: 5 cột
     ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins']

   Categorical features: 3 cột
     ['protocol_type', 'encryption_used', 'browser_type']

   Binary features: 1 cột
     ['unusual_time_access']

   Target feature: 1 cột
     ['attack_detected']



### 2. XỬ LÝ ĐẶC TRƯNG SỐ

In [31]:
numeric_cols = feature_types['Numerical features'].copy()
print(f"\n Các đặc trưng số cần xử lý: {numeric_cols}")
print(f"   (Lưu ý: Các biến nhị phân không cần xử lý theo cách này)")

# 2.1. Thống kê cơ bản trước khi xử lý
print("\n" + "-"*50)
print("2.1. THỐNG KÊ CƠ BẢN TRƯỚC KHI XỬ LÝ")
print("-"*50)

summary_before = []
for col in numeric_cols:
    summary_before.append({
        'Feature': col,
        'Min': df_processed[col].min(),
        'Q1': df_processed[col].quantile(0.25),
        'Median': df_processed[col].median(),
        'Q3': df_processed[col].quantile(0.75),
        'Max': df_processed[col].max(),
        'Mean': df_processed[col].mean(),
        'Std': df_processed[col].std(),
        'Skewness': df_processed[col].skew()
    })

summary_before_df = pd.DataFrame(summary_before)
print(summary_before_df.round(2).to_string(index=False))

# 2.2. Phát hiện và xử lý OUTLIER bằng IQR (cắt về ngưỡng)
print("\n" + "-"*50)
print("2.2. PHÁT HIỆN VÀ XỬ LÝ OUTLIER (Phương pháp IQR)")
print("-"*50)

outlier_report = []
for col in numeric_cols:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Đếm outlier trước khi xử lý
    outliers_before = ((df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)).sum()
    outlier_pct_before = (outliers_before / len(df_processed)) * 100
    
    # Xử lý: cắt (clip) về ngưỡng
    df_processed[col] = df_processed[col].clip(lower_bound, upper_bound)
    
    # Kiểm tra sau xử lý
    outliers_after = ((df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)).sum()
    
    outlier_report.append({
        'Feature': col,
        'Lower bound': round(lower_bound, 2),
        'Upper bound': round(upper_bound, 2),
        'Outliers before': outliers_before,
        'Outliers (%)': f"{outlier_pct_before:.2f}%",
        'Outliers after': outliers_after
    })
    
    print(f"\n   {col}:")
    print(f"     Giới hạn: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"     Outlier: {outliers_before} ({outlier_pct_before:.2f}%) → còn {outliers_after}")

outlier_report_df = pd.DataFrame(outlier_report)
print("\n TỔNG HỢP XỬ LÝ OUTLIER:")
print(outlier_report_df.to_string(index=False))

# 2.3. Kiểm tra phân phối (Skewness) và xử lý bằng LOG TRANSFORM
print("\n" + "-"*50)
print("2.3. XỬ LÝ PHÂN PHỐI LỆCH (LOG TRANSFORM)")
print("-"*50)

# Tính lại skewness sau khi xử lý outlier
print("\n  Skewness sau khi xử lý outlier:")
for col in numeric_cols:
    skew_val = df_processed[col].skew()
    print(f"    {col}: {skew_val:.4f}")

# Tạo cột log cho các cột có skew > 1 (lệch dương mạnh)
log_transformed_cols = []
for col in numeric_cols:
    skew_val = df_processed[col].skew()
    if skew_val > 1:
        # Thêm 1 để tránh log(0) (vì packet_size có thể = 0)
        df_processed[f'{col}_log'] = np.log1p(df_processed[col])
        log_transformed_cols.append(f'{col}_log')
        print(f"\n   Đã tạo {col}_log:")
        print(f"     Skewness trước: {skew_val:.4f}")
        print(f"     Skewness sau: {df_processed[f'{col}_log'].skew():.4f}")
    else:
        print(f"\n   {col}: không cần log transform (skew={skew_val:.4f} < 1)")

# 2.4. Chuẩn hóa (Scaling) các đặc trưng số
print("\n" + "-"*50)
print("2.4. CHUẨN HÓA (SCALING) ĐẶC TRƯNG SỐ")
print("-"*50)

# Chọn các cột sẽ đưa vào scaling
# Bao gồm: cột số gốc (đã xử lý outlier) + cột log (nếu có)
cols_to_scale = numeric_cols.copy()
if log_transformed_cols:
    cols_to_scale.extend(log_transformed_cols)
    print(f"\n   Các cột sẽ được chuẩn hóa: {cols_to_scale}")

# Chọn phương pháp scaling
# - StandardScaler: phù hợp khi dữ liệu có phân phối chuẩn (sau log)
# - RobustScaler: ít bị ảnh hưởng bởi outlier (đã xử lý outlier nên có thể dùng StandardScaler)
print("\n   Sử dụng StandardScaler (chuẩn hóa về mean=0, std=1)")
scaler = StandardScaler()

# Fit và transform
scaled_values = scaler.fit_transform(df_processed[cols_to_scale])

# Tạo tên cột mới
scaled_col_names = [f'{col}_scaled' for col in cols_to_scale]

# Gán vào DataFrame
for i, col in enumerate(scaled_col_names):
    df_processed[col] = scaled_values[:, i]

print(f"\n  Đã tạo {len(scaled_col_names)} cột đã chuẩn hóa: {scaled_col_names}")

# 2.5. Hiển thị thống kê sau khi xử lý
print("\n" + "-"*50)
print("2.5. THỐNG KÊ SAU KHI XỬ LÝ")
print("-"*50)

summary_after = []
for col in cols_to_scale:
    # Cột gốc (đã xử lý outlier)
    if col in df_processed.columns:
        summary_after.append({
            'Feature': col,
            'Min': df_processed[col].min(),
            'Max': df_processed[col].max(),
            'Mean': df_processed[col].mean(),
            'Std': df_processed[col].std(),
            'Skewness': df_processed[col].skew()
        })

summary_after_df = pd.DataFrame(summary_after)
print(summary_after_df.round(4).to_string(index=False))



 Các đặc trưng số cần xử lý: ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins']
   (Lưu ý: Các biến nhị phân không cần xử lý theo cách này)

--------------------------------------------------
2.1. THỐNG KÊ CƠ BẢN TRƯỚC KHI XỬ LÝ
--------------------------------------------------
            Feature  Min     Q1  Median      Q3     Max   Mean    Std  Skewness
network_packet_size 64.0 364.00  498.00  632.00 1285.00 499.56 198.46      0.10
     login_attempts  1.0   3.00    4.00    5.00   13.00   4.03   1.96      0.59
   session_duration  0.5 230.59  547.14 1101.22 7190.39 789.33 789.99      2.08
ip_reputation_score  0.0   0.19    0.31    0.45    0.92   0.33   0.18      0.46
      failed_logins  0.0   1.00    1.00    2.00    5.00   1.52   1.04      0.40

--------------------------------------------------
2.2. PHÁT HIỆN VÀ XỬ LÝ OUTLIER (Phương pháp IQR)
--------------------------------------------------

   network_packet_size:
     Giới 

### 3. XỬ LÝ ĐẶC TRƯNG PHÂN LOẠI

In [32]:
categorical_cols = feature_types['Categorical features'].copy()
print(f"\n Các đặc trưng phân loại cần xử lý: {categorical_cols}")

# 3.1. Phân tích chi tiết từng cột phân loại
print("\n" + "-"*50)
print("3.1. PHÂN TÍCH CHI TIẾT TỪNG CỘT PHÂN LOẠI")
print("-"*50)

categorical_stats = []
for col in categorical_cols:
    unique_vals = df_processed[col].nunique()
    top_values = df_processed[col].value_counts().head(3).to_dict()
    missing = df_processed[col].isna().sum()
    
    categorical_stats.append({
        'Feature': col,
        'Unique values': unique_vals,
        'Most common': top_values,
        'Missing': missing
    })
    
    print(f"\n   {col}:")
    print(f"     - Số giá trị duy nhất: {unique_vals}")
    print(f"     - Phân phối:")
    print(df_processed[col].value_counts().head(5).to_string())
    if 'Unknown' in df_processed[col].values:
        unknown_pct = (df_processed[col] == 'Unknown').mean() * 100
        print(f"     -   Giá trị 'Unknown' chiếm: {unknown_pct:.2f}%")

# 3.2. Quyết định phương pháp mã hóa cho từng cột
print("\n" + "-"*50)
print("3.2. MÃ HÓA CÁC ĐẶC TRƯNG PHÂN LOẠI")
print("-"*50)

# Phương pháp 1: One-Hot Encoding (cho các cột có ít giá trị)
# Phương pháp 2: Label Encoding (cho cột có nhiều giá trị hoặc có thứ tự)
# Phương pháp 3: Target Encoding (nâng cao, có thể dùng sau)

one_hot_cols = []      # Cột sẽ one-hot encoding
label_encoding_map = {} # Cột sẽ label encoding (ánh xạ)

for col in categorical_cols:
    n_unique = df_processed[col].nunique()
    
    if n_unique <= 5:
        # One-hot encoding cho cột có ít giá trị
        one_hot_cols.append(col)
        print(f"\n   {col}: One-Hot Encoding (có {n_unique} giá trị)")
        
        # Thực hiện one-hot encoding
        dummies = pd.get_dummies(df_processed[col], prefix=col, dtype=int)
        df_processed = pd.concat([df_processed, dummies], axis=1)
        
        # In ra các cột mới
        new_cols = [c for c in dummies.columns]
        print(f"     → Đã tạo {len(new_cols)} cột mới: {new_cols}")
        
        # Xóa cột gốc (đã được mã hóa)
        df_processed = df_processed.drop(columns=[col])
        
    else:
        # Label encoding cho cột có nhiều giá trị
        print(f"\n  {col}: Label Encoding (có {n_unique} giá trị)")
        
        # Tạo mapping dựa trên tần suất (giá trị xuất hiện nhiều nhất được gán số nhỏ)
        value_counts = df_processed[col].value_counts()
        label_map = {val: idx for idx, val in enumerate(value_counts.index)}
        label_encoding_map[col] = label_map
        
        # Áp dụng label encoding
        df_processed[f'{col}_encoded'] = df_processed[col].map(label_map)
        
        print(f"Mapping mẫu: {dict(list(label_map.items())[:3])}")
        print(f"Đã tạo cột mới: {col}_encoded")
  

# 3.3. Kiểm tra kết quả mã hóa
print("\n" + "-"*50)
print("3.3. KIỂM TRA KẾT QUẢ MÃ HÓA")
print("-"*50)

print(f"\n   Các cột one-hot đã tạo: {[c for c in df_processed.columns if any(c.startswith(f'{col}_') for col in one_hot_cols)]}")
print(f"   Các cột label encoding đã tạo: {[c for c in df_processed.columns if c.endswith('_encoded')]}")



 Các đặc trưng phân loại cần xử lý: ['protocol_type', 'encryption_used', 'browser_type']

--------------------------------------------------
3.1. PHÂN TÍCH CHI TIẾT TỪNG CỘT PHÂN LOẠI
--------------------------------------------------

   protocol_type:
     - Số giá trị duy nhất: 3
     - Phân phối:
protocol_type
TCP     5235
UDP     1941
ICMP     395

   encryption_used:
     - Số giá trị duy nhất: 2
     - Phân phối:
encryption_used
AES    4706
DES    2865

   browser_type:
     - Số giá trị duy nhất: 5
     - Phân phối:
browser_type
Chrome     4082
Firefox    1527
Edge       1183
Unknown     397
Safari      382
     -   Giá trị 'Unknown' chiếm: 5.24%

--------------------------------------------------
3.2. MÃ HÓA CÁC ĐẶC TRƯNG PHÂN LOẠI
--------------------------------------------------

   protocol_type: One-Hot Encoding (có 3 giá trị)
     → Đã tạo 3 cột mới: ['protocol_type_ICMP', 'protocol_type_TCP', 'protocol_type_UDP']

   encryption_used: One-Hot Encoding (có 2 giá trị)
   

### 4. TẠO ĐẶC TRƯNG MỚI (FEATURE CREATION)

In [33]:
print("\n Tạo các đặc trưng tương tác từ dữ liệu hiện có:")

# 4.1. Tỷ lệ đăng nhập thất bại

df_processed['login_failure_ratio'] = df_processed['failed_logins'] / df_processed['login_attempts']
# Xử lý trường hợp chia cho 0
df_processed['login_failure_ratio'] = df_processed['login_failure_ratio'].fillna(0)
# Giới hạn giá trị tối đa là 1
df_processed['login_failure_ratio'] = df_processed['login_failure_ratio'].clip(0, 1)

print(f"Đã tạo cột: login_failure_ratio")
print(f"Thống kê:")
print(df_processed['login_failure_ratio'].describe().round(4))

# 4.2. Tích hợp điểm uy tín IP và thời gian bất thường

df_processed['ip_risk_score'] = df_processed['ip_reputation_score'] * (df_processed['unusual_time_access'] + 1)
print(f"Đã tạo cột: ip_risk_score")
print(f"Thống kê:")
print(df_processed['ip_risk_score'].describe().round(4))

# 4.3. Mật độ gói tin (packet density)
df_processed['packet_density'] = df_processed['network_packet_size'] / (df_processed['session_duration'] + 1e-6)
print(f"Đã tạo cột: packet_density")
print(f"Thống kê:")
print(df_processed['packet_density'].describe().round(4))

# 4.4. Nhóm tuổi phiên (session_age_group)

def classify_session_duration(duration):
    if duration < 60:
        return 'very_short'
    elif duration < 300:
        return 'short'
    elif duration < 3600:
        return 'medium'
    elif duration < 86400:
        return 'long'
    else:
        return 'very_long'

df_processed['session_age_group'] = df_processed['session_duration'].apply(classify_session_duration)

# One-hot encoding cho nhóm tuổi phiên
session_dummies = pd.get_dummies(df_processed['session_age_group'], prefix='session_age', dtype=int)
df_processed = pd.concat([df_processed, session_dummies], axis=1)
df_processed = df_processed.drop(columns=['session_age_group'])

print(f"Đã tạo các cột: {list(session_dummies.columns)}")



 Tạo các đặc trưng tương tác từ dữ liệu hiện có:
Đã tạo cột: login_failure_ratio
Thống kê:
count    7571.0000
mean        0.4391
std         0.3338
min         0.0000
25%         0.2000
50%         0.3750
75%         0.6667
max         1.0000
Name: login_failure_ratio, dtype: float64
Đã tạo cột: ip_risk_score
Thống kê:
count    7571.0000
mean        0.3800
std         0.2422
min         0.0025
25%         0.2035
50%         0.3403
75%         0.5029
max         1.6966
Name: ip_risk_score, dtype: float64
Đã tạo cột: packet_density
Thống kê:
count    7571.0000
mean        5.2723
std        44.7188
min         0.0266
25%         0.3966
50%         0.8552
75%         2.1129
max      1553.9969
Name: packet_density, dtype: float64
Đã tạo các cột: ['session_age_medium', 'session_age_short', 'session_age_very_short']


### 5. KIỂM TRA CUỐI CÙNG

In [34]:
print("\n5.1. KIỂM TRA GIÁ TRỊ THIẾU (MISSING VALUES)")
missing_counts = df_processed.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
if len(missing_cols) > 0:
    print(f" Vẫn còn {len(missing_cols)} cột có giá trị thiếu:")
    print(missing_cols)
    # Xử lý nốt các giá trị thiếu (nếu có)
    df_processed = df_processed.fillna(0)
    print("Đã điền giá trị 0 cho các ô trống")
else:
    print("Không còn giá trị thiếu nào!")

print("\nKIỂM TRA KIỂU DỮ LIỆU")
print(df_processed.dtypes.value_counts())
print(f"\n  - Số cột số (float64/int64): {len(df_processed.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"  - Số cột object (cần xử lý thêm): {len(df_processed.select_dtypes(include=['object']).columns)}")

# Nếu còn cột object, tiến hành label encoding nốt
object_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"\n Còn cột object: {object_cols}")
    for col in object_cols:
        if col != 'attack_detected':  # Giữ target riêng
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col].astype(str))
            print(f"   Đã label encoding cho cột: {col}")

print("\n5.3. KIỂM TRA KÍCH THƯỚC DỮ LIỆU CUỐI CÙNG")
print(f"  - Số hàng: {df_processed.shape[0]:,}")
print(f"  - Số cột: {df_processed.shape[1]}")
print(f"\n   DANH SÁCH CÁC CỘT CUỐI CÙNG ({len(df_processed.columns)} cột):")
for i, col in enumerate(df_processed.columns, 1):
    print(f"     {i}. {col} ({df_processed[col].dtype})")




5.1. KIỂM TRA GIÁ TRỊ THIẾU (MISSING VALUES)
Không còn giá trị thiếu nào!

KIỂM TRA KIỂU DỮ LIỆU
int64      17
float64    13
Name: count, dtype: int64

  - Số cột số (float64/int64): 30
  - Số cột object (cần xử lý thêm): 0

5.3. KIỂM TRA KÍCH THƯỚC DỮ LIỆU CUỐI CÙNG
  - Số hàng: 7,571
  - Số cột: 30

   DANH SÁCH CÁC CỘT CUỐI CÙNG (30 cột):
     1. network_packet_size (int64)
     2. login_attempts (int64)
     3. session_duration (float64)
     4. ip_reputation_score (float64)
     5. failed_logins (float64)
     6. unusual_time_access (int64)
     7. attack_detected (int64)
     8. session_duration_log (float64)
     9. network_packet_size_scaled (float64)
     10. login_attempts_scaled (float64)
     11. session_duration_scaled (float64)
     12. ip_reputation_score_scaled (float64)
     13. failed_logins_scaled (float64)
     14. session_duration_log_scaled (float64)
     15. protocol_type_ICMP (int64)
     16. protocol_type_TCP (int64)
     17. protocol_type_UDP (int64)
     18.

### 6. LƯU DỮ LIỆU

In [35]:
# Đảm bảo cột target attack_detected là kiểu số (int)
if 'attack_detected' in df_processed.columns:
    df_processed['attack_detected'] = df_processed['attack_detected'].astype(int)

# Lưu dữ liệu
output_path = "../data_processed/feature_engineered_data.parquet"
df_processed.to_parquet(output_path, index=False)
print(f"\n Đã lưu dữ liệu đã xử lý đặc trưng vào: {output_path}")

# Cũng lưu dưới dạng CSV để dễ xem
csv_path = "../data_processed/feature_engineered_data.csv"
df_processed.to_csv(csv_path, index=False)
print(f" Đã lưu dưới dạng CSV: {csv_path}")



 Đã lưu dữ liệu đã xử lý đặc trưng vào: ../data_processed/feature_engineered_data.parquet
 Đã lưu dưới dạng CSV: ../data_processed/feature_engineered_data.csv


## **KẾT THÚC PHẦN XỬ LÝ ĐẶC TRƯNG (FEATURE ENGINEERING)**